#Phase 1: Environment Setup & Drive Integration

In [ ]:
# =========================
# PHASE 1: ENV SETUP
# =========================

# -------- 1. Mount Drive --------
from google.colab import drive
drive.mount('/content/drive')

# -------- 2. Imports --------
import os
import torch
import random
import numpy as np
import logging

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# -------- 3. Config Class --------
class Config:
    # Project root
    PROJECT_ROOT = "/content/drive/MyDrive/MHQA_Project"

    # Data paths
    DATA_DIR = os.path.join(PROJECT_ROOT, "data")
    MHQA_PATH = os.path.join(DATA_DIR, "mhqa.csv")
    MHQA_B_PATH = os.path.join(DATA_DIR, "mhqa-b.csv")

    # Output directories
    MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
    CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
    METRICS_DIR = os.path.join(PROJECT_ROOT, "metrics")
    PLOTS_DIR = os.path.join(PROJECT_ROOT, "plots")
    LOGS_DIR = os.path.join(PROJECT_ROOT, "logs")
    RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

    # Seed
    SEED = 42

# -------- 4. Create Folder Structure --------
def create_directories():
    dirs = [
        Config.DATA_DIR,
        Config.MODELS_DIR,
        Config.CHECKPOINTS_DIR,
        Config.METRICS_DIR,
        Config.PLOTS_DIR,
        Config.LOGS_DIR,
        Config.RESULTS_DIR
    ]

    for d in dirs:
        os.makedirs(d, exist_ok=True)

    print("✅ Folder structure ready!")

# -------- 5. Check Dataset Paths --------
def verify_data():
    print("📂 Checking dataset paths...")
    print("MHQA exists:", os.path.exists(Config.MHQA_PATH))
    print("MHQA-B exists:", os.path.exists(Config.MHQA_B_PATH))

# -------- 6. Set Device --------
def setup_device():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"⚡ Using device: {device}")
    return device

# -------- 7. Set Seed --------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print("🌱 Seed set!")

# -------- 8. Setup Logging --------
def setup_logging():
    log_path = os.path.join(Config.LOGS_DIR, "project.log")

    logging.basicConfig(
        filename=log_path,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s"
    )

    logging.info("Project initialized")
    print("📝 Logging initialized!")

# -------- 9. Run Everything --------
def main():
    print("\n🚀 Initializing Phase 1...\n")

    create_directories()
    verify_data()
    device = setup_device()
    set_seed(Config.SEED)
    setup_logging()

    print("\n✅ Phase 1 Complete!\n")
    return device

# Execute
device = main()

#Phase 2 : Library Setup + Config Expansion + Model Registry

In [ ]:
# =========================
# PHASE 2: LIBRARIES + CONFIG + MODEL REGISTRY
# =========================

# -------- 1. Core Imports --------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# -------- 2. Sklearn --------
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef, log_loss
)

# -------- 3. Transformers --------
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

# -------- 4. Torch Utils --------
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# -------- 5. Update Config Class --------
class TrainConfig:
    MAX_LEN = 256
    BATCH_SIZE = 16
    EPOCHS = 2
    LR = 2e-5

    NUM_LABELS = 4  # anxiety, depression, trauma, OCD

    # For reproducibility
    SEED = 42

# -------- 6. Model Registry --------
# (HuggingFace model names mapped to simple keys)

MODEL_REGISTRY = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "distilbert": "distilbert-base-uncased",
    "electra": "google/electra-base-discriminator"
}

print("📦 Total models registered:", len(MODEL_REGISTRY))

# -------- 7. Model Loader Function --------
def load_model_and_tokenizer(model_key):
    model_name = MODEL_REGISTRY[model_key]

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=TrainConfig.NUM_LABELS
    )

    model.to(device)

    return tokenizer, model

# -------- 8. Optimizer & Scheduler --------
def get_optimizer_scheduler(model, train_loader_len):
    optimizer = AdamW(model.parameters(), lr=TrainConfig.LR)

    total_steps = train_loader_len * TrainConfig.EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )

    return optimizer, scheduler

# -------- 9. Quick Sanity Check --------
print("✅ Phase 2 setup complete!")
print("Example model keys:", list(MODEL_REGISTRY.keys()))

#Phase 3 : Data Loading + Cleaning + Label Encoding + EDA

In [ ]:
# =========================
# PHASE 3: DATA LOADING + PREPROCESSING + EDA (FINAL FINAL)
# =========================

print("📥 Loading datasets...")

mhqa_df = pd.read_csv(Config.MHQA_PATH)
mhqa_b_df = pd.read_csv(Config.MHQA_B_PATH)

print("✅ Data loaded!")
print("MHQA shape:", mhqa_df.shape)
print("MHQA-B shape:", mhqa_b_df.shape)


# -------- 1. Inspect Columns --------
print("\n📊 Columns:")
print(mhqa_df.columns)


# -------- 2. Define Columns --------
TEXT_COLUMN = "question"
LABEL_COLUMN = "topic"


# -------- 3. Keep Required Columns --------
mhqa_df = mhqa_df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()
mhqa_b_df = mhqa_b_df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()

print("\n✅ After cleaning:")
print("MHQA:", mhqa_df.shape)
print("MHQA-B:", mhqa_b_df.shape)


# -------- 4. Basic Text Cleaning --------
def clean_text(text):
    return str(text).strip().lower()

mhqa_df[TEXT_COLUMN] = mhqa_df[TEXT_COLUMN].apply(clean_text)
mhqa_b_df[TEXT_COLUMN] = mhqa_b_df[TEXT_COLUMN].apply(clean_text)


# -------- 5. Label Encoding --------
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Fit on large dataset
label_encoder.fit(mhqa_b_df[LABEL_COLUMN])

# Transform both
mhqa_df[LABEL_COLUMN] = label_encoder.transform(mhqa_df[LABEL_COLUMN])
mhqa_b_df[LABEL_COLUMN] = label_encoder.transform(mhqa_b_df[LABEL_COLUMN])

print("\n🏷 Label Mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} → {i}")


# -------- 6. Save Label Encoder --------
import pickle

with open(os.path.join(Config.MODELS_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)

print("💾 Label encoder saved!")


# -------- 7. Text Length Analysis (MHQA) --------
mhqa_df["text_length"] = mhqa_df[TEXT_COLUMN].apply(lambda x: len(x.split()))

plt.figure()
mhqa_df["text_length"].hist()
plt.title("Text Length Distribution (MHQA)")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.savefig(os.path.join(Config.PLOTS_DIR, "mhqa_text_length_distribution.png"))
plt.show()


# -------- Text Length Analysis (MHQA-B) --------
mhqa_b_df["text_length"] = mhqa_b_df[TEXT_COLUMN].apply(lambda x: len(x.split()))

plt.figure()
mhqa_b_df["text_length"].hist()
plt.title("Text Length Distribution (MHQA-B)")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.savefig(os.path.join(Config.PLOTS_DIR, "mhqa_b_text_length_distribution.png"))
plt.show()



# -------- 8. Class Distribution (MHQA) --------
plt.figure()
sns.countplot(x=mhqa_df[LABEL_COLUMN])
plt.title("Class Distribution (MHQA)")
plt.savefig(os.path.join(Config.PLOTS_DIR, "mhqa_class_distribution.png"))
plt.show()


# -------- 9. Class Distribution (MHQA-B) --------
plt.figure()
sns.countplot(x=mhqa_b_df[LABEL_COLUMN])
plt.title("Class Distribution (MHQA-B)")
plt.savefig(os.path.join(Config.PLOTS_DIR, "mhqa_b_class_distribution.png"))
plt.show()


# -------- 10. Save Data Summary --------
summary_path = os.path.join(Config.RESULTS_DIR, "data_summary.txt")

with open(summary_path, "w") as f:
    f.write(f"MHQA Shape: {mhqa_df.shape}\n")
    f.write(f"MHQA-B Shape: {mhqa_b_df.shape}\n\n")

    f.write("Label Mapping:\n")
    for i, label in enumerate(label_encoder.classes_):
        f.write(f"{label} → {i}\n")

print("📝 Data summary saved!")


# -------- 11. Final Output --------
print("\n✅ Phase 3 Complete!")

# PHASE 4: DATASET + TOKENIZATION + DATALOADER + CLASS WEIGHTS

In [ ]:
# =========================
# PHASE 4: DATASET + TOKENIZATION + DATALOADER + CLASS WEIGHTS
# =========================

print("🚀 Starting Phase 4...")


# -------- 1. Train / Validation Split --------
train_df = mhqa_b_df.copy()

val_df = mhqa_df.sample(frac=0.5, random_state=42)
test_df = mhqa_df.drop(val_df.index)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)


# -------- 2. Custom Dataset --------
class MHQADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=TrainConfig.MAX_LEN,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# -------- 3. Load Tokenizer (example: BERT) --------
tokenizer, _ = load_model_and_tokenizer("bert")


# -------- 4. Create Datasets --------
train_dataset = MHQADataset(
    train_df["question"],
    train_df["topic"],
    tokenizer,
    TrainConfig.MAX_LEN
)

val_dataset = MHQADataset(
    val_df["question"],
    val_df["topic"],
    tokenizer,
    TrainConfig.MAX_LEN
)

test_dataset = MHQADataset(
    test_df["question"],
    test_df["topic"],
    tokenizer,
    TrainConfig.MAX_LEN
)


# -------- 5. DataLoaders --------
train_loader = DataLoader(
    train_dataset,
    batch_size=TrainConfig.BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TrainConfig.BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TrainConfig.BATCH_SIZE,
    shuffle=False
)

print("✅ DataLoaders ready!")


# -------- 6. Class Weights (IMPORTANT) --------
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df["topic"]),
    y=train_df["topic"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("⚖️ Class Weights:", class_weights)


# -------- 7. Loss Function --------
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)


# -------- 8. Quick Batch Check --------
batch = next(iter(train_loader))

print("\n🔍 Batch check:")
print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)


# -------- 9. Done --------
print("\n✅ Phase 4 Complete!")

#Phase 5 — Training Loop (with tqdm + logging + saving)

In [ ]:
# =========================
# PHASE 5: FINAL PIPELINE (ELECTRA ADDED, DEBERTA REMOVED)
# =========================

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef, log_loss
)
from sklearn.preprocessing import label_binarize

from torch.amp import autocast, GradScaler

print("🚀 Starting FINAL PIPELINE (with ELECTRA)...")

# -------- FIX LABELS --------
for df in [train_df, val_df, test_df]:
    if df["topic"].min() != 0:
        df["topic"] = df["topic"] - df["topic"].min()

print("Labels:", sorted(train_df["topic"].unique()))

# -------- SPEED SETTINGS --------
SUBSET_SIZE = 15000
train_df_fast = train_df.sample(n=min(SUBSET_SIZE, len(train_df)), random_state=42)

def get_train_loader(tokenizer):
    dataset = MHQADataset(
        train_df_fast["question"],
        train_df_fast["topic"],
        tokenizer,
        max_len=128
    )
    return DataLoader(dataset, batch_size=32, shuffle=True)

scaler = GradScaler("cuda")

models_to_train = ["electra", "bert", "roberta", "distilbert"]

all_results = {}

# -------- TRAIN FUNCTION --------
def train_model(model, name, train_loader):

    optimizer, scheduler = get_optimizer_scheduler(model, len(train_loader))

    best_val = float('inf')
    train_losses, val_losses = [], []

    for epoch in range(2):
        print(f"\n🔥 {name.upper()} Epoch {epoch+1}")

        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"{name} Training"):
            optimizer.zero_grad()

            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # ELECTRA + others → AMP safe
            with autocast("cuda"):
                outputs = model(input_ids=ids, attention_mask=mask)
                loss = loss_fn(outputs.logits.float(), labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        train_losses.append(train_loss)

        # -------- VALIDATION --------
        model.eval()
        val_loss_total = 0

        with torch.no_grad():
            for batch in val_loader:
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(input_ids=ids, attention_mask=mask)
                val_loss_total += loss_fn(outputs.logits.float(), labels).item()

        val_loss = val_loss_total / len(val_loader)
        val_losses.append(val_loss)

        print(f"Train {train_loss:.4f} | Val {val_loss:.4f}")

        model_path = os.path.join(Config.MODELS_DIR, f"{name}.pth")
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), model_path)

    return train_losses, val_losses


# -------- EVALUATE FUNCTION --------
def evaluate_model(model, name):

    model.eval()
    preds, labels_list, probs = [], [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"{name} Testing"):
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=ids, attention_mask=mask)
            prob = torch.softmax(outputs.logits.float(), dim=1)
            pred = torch.argmax(prob, dim=1)

            preds.extend(pred.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
            probs.extend(prob.cpu().numpy())

    preds = np.array(preds)
    labels_list = np.array(labels_list)
    probs = np.array(probs)

    metrics = {
        "accuracy": float(accuracy_score(labels_list, preds)),
        "precision": float(precision_score(labels_list, preds, average="macro")),
        "recall": float(recall_score(labels_list, preds, average="macro")),
        "f1_score": float(f1_score(labels_list, preds, average="macro")),
        "mcc": float(matthews_corrcoef(labels_list, preds)),
        "log_loss": float(log_loss(labels_list, probs))
    }

    y_bin = label_binarize(labels_list, classes=np.unique(labels_list))
    metrics["roc_auc"] = float(roc_auc_score(y_bin, probs, multi_class='ovr'))

    # Confusion Matrix
    plt.figure(figsize=(6,5))
    sns.heatmap(confusion_matrix(labels_list, preds), annot=True, fmt="d")
    plt.title(f"{name} Confusion Matrix")
    plt.savefig(os.path.join(Config.PLOTS_DIR, f"{name}_cm.png"))
    plt.close()

    # ROC Curve
    plt.figure(figsize=(6,5))
    for i in range(len(np.unique(labels_list))):
        plt.plot(y_bin[:, i], probs[:, i], label=f"Class {i}")
    plt.legend()
    plt.title(f"{name} ROC Curve")
    plt.savefig(os.path.join(Config.PLOTS_DIR, f"{name}_roc.png"))
    plt.close()

    with open(os.path.join(Config.METRICS_DIR, f"{name}.json"), "w") as f:
        json.dump(metrics, f, indent=4)

    return metrics


# -------- MAIN LOOP --------
for name in models_to_train:

    print(f"\n🚀 {name.upper()}")

    tokenizer, model = load_model_and_tokenizer(name)
    train_loader = get_train_loader(tokenizer)

    # TRAIN
    train_losses, val_losses = train_model(model, name, train_loader)

    # LOSS CURVE
    plt.figure(figsize=(6,5))
    plt.plot(train_losses, label="Train")
    plt.plot(val_losses, label="Val")
    plt.legend()
    plt.title(f"{name} Loss Curve")
    plt.savefig(os.path.join(Config.PLOTS_DIR, f"{name}_loss.png"))
    plt.close()

    # LOAD BEST MODEL
    model.load_state_dict(torch.load(os.path.join(Config.MODELS_DIR, f"{name}.pth")))
    model.to(device)

    # EVALUATE
    all_results[name] = evaluate_model(model, name)

    print(f"✅ {name.upper()} DONE")


# -------- SAVE COMPARISON --------
with open(os.path.join(Config.RESULTS_DIR, "comparison.json"), "w") as f:
    json.dump(all_results, f, indent=4)

print("\n🏁 ALL DONE SUCCESSFULLY!")

# PHASE 6: MODEL COMPARISON + ANALYSIS

In [ ]:
# =========================
# PHASE 6: FINAL VISUAL ANALYTICS
# =========================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

sns.set(style="whitegrid")

print("🚀 Starting Phase 6: Visual Analytics")

# -------------------------
# LOAD METRICS FROM JSON
# -------------------------
results = []

for file in os.listdir(Config.METRICS_DIR):
    if file.endswith(".json"):
        with open(os.path.join(Config.METRICS_DIR, file), "r") as f:
            data = json.load(f)

        model_name = file.replace(".json", "")
        data["model"] = model_name
        results.append(data)

df = pd.DataFrame(results)

# Keep relevant columns
df = df[["model", "accuracy", "precision", "recall", "f1_score", "roc_auc", "mcc"]]

df = df.rename(columns={"f1_score": "f1"})
df = df.sort_values(by="f1", ascending=False).reset_index(drop=True)

print("\n📊 Models Loaded:")
display(df)

# Save CSV
df.to_csv(os.path.join(Config.RESULTS_DIR, "all_models_results.csv"), index=False)


# -------------------------
# COLOR PALETTE
# -------------------------
muted_colors = [
    "#6F9F95", "#7F8FB0", "#C08A86",
    "#8FAF7A", "#AFAF98", "#8C7A8F"
]

def adjust_color(color, factor):
    c = np.array(mcolors.to_rgb(color))
    return np.clip(c * factor, 0, 1)


# -------------------------
# INDIVIDUAL METRIC PLOTS
# -------------------------
def fancy_bar(metric):

    plt.figure(figsize=(10,5))

    sns.barplot(
        x="model",
        y=metric,
        data=df,
        palette=muted_colors[:len(df)]
    )

    for i, v in enumerate(df[metric]):
        plt.text(i, v + 0.01, f"{v*100:.2f}%", ha='center', fontsize=9)

    plt.title(f"{metric.upper()} Comparison", fontsize=14, weight='bold')
    plt.xticks(rotation=30)

    plt.tight_layout()
    plt.savefig(os.path.join(Config.PLOTS_DIR, f"{metric}_comparison.png"))
    plt.close()


for metric in ["accuracy", "precision", "recall", "f1"]:
    fancy_bar(metric)


# -------------------------
# GROUPED BAR PLOT
# -------------------------
plt.figure(figsize=(14,6))

metrics = ["accuracy", "precision", "recall", "f1"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1 Score"]

bar_width = 0.18
x = np.arange(len(df))

shade_factors = [1.0, 1.15, 0.85, 0.7]

for i, row in df.iterrows():

    base_color = muted_colors[i % len(muted_colors)]

    values = [row[m] for m in metrics]

    for j, v in enumerate(values):

        color = adjust_color(base_color, shade_factors[j])

        plt.bar(
            x[i] + j * bar_width,
            v,
            width=bar_width,
            color=color
        )

        plt.text(
            x[i] + j * bar_width,
            v + 0.01,
            f"{v*100:.1f}%",
            ha='center',
            fontsize=8,
            rotation=45
        )

plt.xticks(x + bar_width * 1.5, df["model"], rotation=30)
plt.title("All Models vs All Metrics", fontsize=14, weight='bold')
plt.ylabel("Score")

legend_elements = [
    Patch(facecolor=adjust_color("#888888", shade_factors[i]), label=metric_labels[i])
    for i in range(4)
]

plt.legend(handles=legend_elements, title="Metrics")

plt.tight_layout()
plt.savefig(os.path.join(Config.PLOTS_DIR, "grouped_metrics.png"))
plt.close()


# -------------------------
# HEATMAP
# -------------------------
plt.figure(figsize=(10,5))

heat_df = df.set_index("model")[["accuracy", "precision", "recall", "f1"]]

sns.heatmap(
    heat_df,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Model Performance Heatmap", fontsize=14, weight='bold')

plt.tight_layout()
plt.savefig(os.path.join(Config.PLOTS_DIR, "heatmap.png"))
plt.close()


# -------------------------
# FINAL RANKING
# -------------------------
ranking = df.copy()
ranking["Rank"] = ranking.index + 1

print("\n🏆 FINAL MODEL RANKING:")
display(ranking)

ranking.to_csv(os.path.join(Config.RESULTS_DIR, "final_model_ranking.csv"), index=False)


# -------------------------
# SUMMARY
# -------------------------
best = ranking.iloc[0]

summary = f"""
BEST MODEL: {best['model']}

Accuracy: {best['accuracy']:.4f}
Precision: {best['precision']:.4f}
Recall: {best['recall']:.4f}
F1 Score: {best['f1']:.4f}
ROC-AUC: {best['roc_auc']:.4f}
MCC: {best['mcc']:.4f}
"""

print(summary)

with open(os.path.join(Config.RESULTS_DIR, "final_summary.txt"), "w") as f:
    f.write(summary)

print("\n🏁 Phase 6 Complete!")

# PHASE 7: BERT INFERENCE (SAMPLE PREDICTIONS)


In [ ]:
# =========================
# PHASE 7: BERT INFERENCE + SAVE JSON
# =========================

import json

print("🧠 Running BERT on sample questions...\n")

# -------- LABEL MAPPING --------
label_map = {
    0: "Anxiety",
    1: "Depression",
    2: "Trauma",
    3: "OCD"
}

# -------- LOAD MODEL --------
tokenizer, model = load_model_and_tokenizer("bert")

model_path = os.path.join(Config.MODELS_DIR, "bert.pth")
model.load_state_dict(torch.load(model_path))
model.to(device)
model.eval()

# -------- SAMPLE DATA --------
sample_df = test_df.sample(10).reset_index(drop=True)

results = []

# -------- INFERENCE --------
for i in range(len(sample_df)):

    question = sample_df.loc[i, "question"]
    actual_label = int(sample_df.loc[i, "topic"])

    inputs = tokenizer(
        question,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits.float(), dim=1)
        pred = torch.argmax(probs, dim=1).item()

    result = {
        "question": question,
        "actual_domain": label_map[actual_label],
        "predicted_domain": label_map[pred]
    }

    results.append(result)

    # PRINT
    print(f"\n\n\n🔹 Example {i+1}")
    print(f"Question: {question}\n")
    print(f"Actual: {label_map[actual_label]}\n")
    print(f"Predicted: {label_map[pred]}\n\n\n")
    print("-" * 80)


# -------- SAVE TO JSON --------
save_path = os.path.join(Config.RESULTS_DIR, "sample_predictions.json")

with open(save_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"\n💾 Saved predictions to: {save_path}")